# Converting ToponymExtractor outputs at word level

In [ ]:
# Imports
from typing import Final
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from os import environ
from pickle import load as load_pickle

# Constants and presets
PROJECT_DIR: Final[Path]
PROJECT_DIR = Path(find_dotenv(".env", 1, 1)).absolute().parent
environ["PROJECT_DIR"] = str(PROJECT_DIR)
load_dotenv(PROJECT_DIR.joinpath(".env"))

In [ ]:
from json import load as load_json
from geopandas import read_file
from outputs import\
    convert_ToponymExtractor_outputs_to_gdf, read_pickle_queue

In [ ]:
from project_utils import parse_path

In [ ]:
pred_fp = parse_path(
    r"outputs/manual-labelling/merge-trial/merge-trial-outputs/mt-outputs-word.pkl",
    "LOCAL_DIR"
)
gcp_fp = parse_path(
    r"outputs/manual-labelling/merge-trial/manual-split/control-points.gpkg",
    "LOCAL_DIR"
)
save_geo_fp = parse_path(
    r"outputs/manual-labelling/merge-trial/word-preds.gpkg",
    "LOCAL_DIR"
)
save_err_fp = parse_path(
    r"outputs/manual-labelling/merge-trial/word-pred-errs.csv",
    "LOCAL_DIR"
)

In [ ]:
predictions = read_pickle_queue(pred_fp)
ctrl_points = read_file(gcp_fp)

In [ ]:
print(*predictions[0]["words"][0].keys(), sep = "\n")

In [ ]:
from numpy import array, clip
from shapely import Polygon
from pandas import DataFrame
from geopandas import GeoDataFrame
from rasterio.transform import GCPTransformer, AffineTransformer
from edina import get_transformer_from_geodataframe
from outputs import normalize_geometries

png_w, png_h = 750, 750

pngs, errors, data = set(), [], []
for image in predictions:
    # Check record has not been seen before
    if image["image"] not in pngs:
        # Update seen pngs log
        pngs.add(image["image"])

        for i, word in enumerate(image["words"]):
            # some records errored out - so only upack those with expected
            # formats
            if "error" not in word:
                # Create record
                record = {
                    "png_filename": image["image"],
                    "center_bezier_pts": word["center_bezier_pts"],
                    "avg_height": word["avg_height"],
                    "word": word["text"],
                    "score": word["score"]
                }
                # Clip vertices to within the bounds of image
                vertices = array([
                    [*el]
                    for el
                    in zip(word["polygon_x"], word["polygon_y"])
                ])
                vertices[:, 0] =\
                    clip(vertices[:, 0], 0., float(png_w - 1))
                vertices[:, 1] =\
                    clip(vertices[:, 1], 0., float(png_h - 1))
                vertices = vertices.tolist()

                # Get georeference control points transformer
                gcp_trans = ctrl_points.loc[
                    ctrl_points["png_filename"] == image["image"]
                ]
                if len(gcp_trans):
                    gcp_trans =\
                        get_transformer_from_geodataframe(gcp_trans)
                    # Convert pixel location coordinates to latitude/
                    # longitude and create geometry field
                    record["geometry"] = Polygon([
                        gcp_trans.xy(r, c) for c, r in vertices
                    ])
                    # Documentation recommends calling close on
                    # GCPTransformer after calling transforms
                    gcp_trans.close()
                else:
                    print(
                        f"Georeferencing control points not found "\
                        f"for image {image["image"]}"
                    )
                    record["geometry"] = None
                
                # Add record to data
                data.append(record)

            else:
                # Error handling
                errors.append({
                    "png_filename": image["image"], "error": image["error"]
                })

data = GeoDataFrame(data, crs = ctrl_points.crs)
data["geometry"] = normalize_geometries(data["geometry"])
errors = DataFrame(errors)

In [ ]:
errors.to_csv(save_err_fp)
data.to_file(save_geo_fp)